https://karan3-zoh.medium.com/paper-summary-imagenet-classification-with-deep-convolutional-neural-networks-41ce6c65960

In [103]:
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

In [104]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=96, kernel_size=(11,11), stride=4)
        self.conv2 = nn.Conv2d(in_channels=96, out_channels=256, kernel_size=(5,5), padding=2)
        self.conv3 = nn.Conv2d(in_channels=256, out_channels=384, kernel_size=(3,3), padding=1)
        self.conv4 = nn.Conv2d(in_channels=384, out_channels=384, kernel_size=(3,3), padding=1)
        self.conv5 = nn.Conv2d(in_channels=384, out_channels=256, kernel_size=(3,3), padding=1)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(in_features=9216, out_features=4096)
        self.fc2 = nn.Linear(in_features=4096, out_features=4096)
        self.fc3 = nn.Linear(4096, 1)
    
    def forward(self, x):
        x = self.conv1(x)
        x = F.max_pool2d(x, kernel_size=(3,3), stride=2)
        x = self.conv2(x)
        x = F.max_pool2d(x, kernel_size=(3,3), stride=2)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = F.max_pool2d(x, kernel_size=(3,3), stride=2)
        x = self.flatten(x)
        print(x.shape)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        print(x.shape)
        return x
        


In [105]:
model = AlexNet()

In [106]:
dummy = torch.tensor(np.zeros((16,3,  227,227))).float()

summary(model=model, input_data=dummy)

torch.Size([16, 9216])
torch.Size([16, 1])


Layer (type:depth-idx)                   Output Shape              Param #
AlexNet                                  [16, 1]                   --
├─Conv2d: 1-1                            [16, 96, 55, 55]          34,944
├─Conv2d: 1-2                            [16, 256, 27, 27]         614,656
├─Conv2d: 1-3                            [16, 384, 13, 13]         885,120
├─Conv2d: 1-4                            [16, 384, 13, 13]         1,327,488
├─Conv2d: 1-5                            [16, 256, 13, 13]         884,992
├─Flatten: 1-6                           [16, 9216]                --
├─Linear: 1-7                            [16, 4096]                37,752,832
├─Linear: 1-8                            [16, 4096]                16,781,312
├─Linear: 1-9                            [16, 1]                   4,097
Total params: 58,285,441
Trainable params: 58,285,441
Non-trainable params: 0
Total mult-adds (G): 18.11
Input size (MB): 9.89
Forward/backward pass size (MB): 84.26
Params size (M

In [107]:
data = pd.read_csv("../data/chest_xray/chest_xray_dataset.csv")

In [108]:
train_data = data[data['split'] == 'train']

In [109]:
train_data.iloc[0]['path']

'data/chest_xray/train/NORMAL/NORMAL2-IM-0983-0001-0002.jpeg'

In [110]:
from PIL import Image
import os 

class XrayDataset(Dataset):
    def __init__(self, split, transform):
        if split == "train":
            self.data = data[data['split'] == 'train']

        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        data_item = self.data.iloc[idx]
        label = self.data.iloc[idx]['class']
        image_data = Image.open(os.path.join("..", data_item['path']))
        print(np.array(image_data).shape)
        image_data = self.transform(image_data)
        return image_data, label

In [ ]:
train_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.uint8, scale=True),
    v2.RandomResizedCrop(size=(500, 500)),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.uint8, scale=True),
    v2.CenterCrop(size=(500, 500)),
    v2.ToDtype(torch.float32, scale=True),
])

In [112]:
train_dataset = XrayDataset(split='train', transform=train_transform)

In [113]:
train_dataloader = DataLoader(dataset=train_dataset, shuffle=True, batch_size=16)

In [ ]:
rr